# Session 6 — Regression bake-off

I'm opening the regression bake-off. First I reload `features.parquet` and rebuild
the leakage-safe split straight from the crystallized `ModelTrainer`, then re-confirm
the persistence bar my 9 models must beat — computed live, never hardcoded.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src import config
from src.models.baseline import PersistenceBaseline
from src.models.trainer import ModelTrainer

# The model-ready feature table produced in Session 4.
features = pd.read_parquet(config.FEATURES_PATH)

# Rebuild the leakage-safe split from the crystallized trainer (not inline).
trainer = ModelTrainer()
train, test = trainer.split(features)

# Re-establish the LIVE bar every regression model must beat.
baseline = PersistenceBaseline()
bar = baseline.regression_scores(test)

print("features:", features.shape)
print("train / test rows:", len(train), "/", len(test))
print("baseline (test):", {k: round(v, 3) for k, v in bar.items()})

features: (201725, 30)
train / test rows: 155131 / 46594
baseline (test): {'mae': np.float64(24.008), 'rmse': np.float64(41.404), 'r2': 0.745}


## Bucket 1 — The 9-model roster

I line up 9 regressors across four families — linear, distance, tree, and boosting —
so the bake-off compares fundamentally different ways of reading the clues. I seed only
the stochastic models, and I bench SVR on purpose: it's too slow on 155k rows.

In [2]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# The 9-model regression roster, grouped by family.
# random_state only on the stochastic models; the rest are deterministic.
regressors = [
    ("LinearRegression", LinearRegression()),
    ("Ridge",            Ridge()),
    ("Lasso",            Lasso()),
    ("ElasticNet",       ElasticNet()),
    ("KNN",              KNeighborsRegressor(n_jobs=-1)),
    ("DecisionTree",     DecisionTreeRegressor(random_state=config.RANDOM_STATE)),
    ("RandomForest",     RandomForestRegressor(random_state=config.RANDOM_STATE, n_jobs=-1)),
    ("XGBoost",          XGBRegressor(random_state=config.RANDOM_STATE, n_jobs=-1)),
    ("LightGBM",         LGBMRegressor(random_state=config.RANDOM_STATE, n_jobs=-1, verbose=-1)),
]

# Sanity: 9 chefs, and each wraps into the leakage-safe pipeline without error.
print("roster size:", len(regressors))
for name, model in regressors:
    pipe = trainer.make_pipeline(model)
    print(f"  {name:16s} -> wrapped ok ({type(model).__name__})")

roster size: 9
  LinearRegression -> wrapped ok (LinearRegression)
  Ridge            -> wrapped ok (Ridge)
  Lasso            -> wrapped ok (Lasso)
  ElasticNet       -> wrapped ok (ElasticNet)
  KNN              -> wrapped ok (KNeighborsRegressor)
  DecisionTree     -> wrapped ok (DecisionTreeRegressor)
  RandomForest     -> wrapped ok (RandomForestRegressor)
  XGBoost          -> wrapped ok (XGBRegressor)
  LightGBM         -> wrapped ok (LGBMRegressor)


## Bucket 2 — Cross-validated bake-off

I run all 9 models through the trainer's 5-fold TimeSeriesSplit on the TRAINING set only —
the test set stays sealed until I've picked my finalists. Each model is scored on 5 "future"
windows; I average MAE/RMSE/R2 into one leaderboard. sklearn reports errors as negative
(higher=better), so I flip the sign back when reading.

In [3]:
import pandas as pd
from sklearn.model_selection import cross_validate

# X = the model's feature contract; y = tomorrow's AQI. Train frame is already
# date-sorted by trainer.split(), so TimeSeriesSplit cuts by TIME, not by city.
X_train = train[config.MODEL_FEATURES]
y_train = train[config.TARGET_AQI_COL]

cv = trainer.get_cv()   # TimeSeriesSplit, 5 expanding-window folds

# sklearn convention: errors are reported NEGATED (higher = better). We flip back.
scoring = {
    "mae":  "neg_mean_absolute_error",
    "rmse": "neg_root_mean_squared_error",
    "r2":   "r2",
}

rows = []
for name, model in regressors:
    pipe = trainer.make_pipeline(model)
    result = cross_validate(
        pipe, X_train, y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=1,             # avoid nested parallelism (see note below)
        error_score="raise",  # fail loudly, never silently score NaN
    )
    rows.append({
        "model":   name,
        "cv_mae":  -result["test_mae"].mean(),   # flip the sign back
        "cv_rmse": -result["test_rmse"].mean(),
        "cv_r2":    result["test_r2"].mean(),
        "fit_sec":  result["fit_time"].mean(),
    })
    print(f"  done: {name:16s} cv_mae={rows[-1]['cv_mae']:.3f}")

leaderboard = pd.DataFrame(rows).sort_values("cv_mae").reset_index(drop=True)
print("\npersistence bar (test-set reference): mae 24.008\n")
leaderboard.round(3)

  done: LinearRegression cv_mae=25.824
  done: Ridge            cv_mae=25.820
  done: Lasso            cv_mae=27.759
  done: ElasticNet       cv_mae=29.747
  done: KNN              cv_mae=30.770
  done: DecisionTree     cv_mae=38.612
  done: RandomForest     cv_mae=26.602
  done: XGBoost          cv_mae=27.084
  done: LightGBM         cv_mae=26.047

persistence bar (test-set reference): mae 24.008



,model,cv_mae,cv_rmse,cv_r2,fit_sec
0,Ridge,25.820,41.016,0.764,0.197
1,LinearRegression,25.824,41.018,0.764,0.256
2,LightGBM,26.047,41.232,0.762,0.510
3,RandomForest,26.602,41.793,0.755,21.016
4,XGBoost,27.084,42.766,0.744,0.849
5,Lasso,27.759,42.911,0.744,1.390
6,ElasticNet,29.747,44.601,0.724,0.513
7,KNN,30.770,47.206,0.691,0.224
8,DecisionTree,38.612,61.476,0.473,4.848


## Bucket 3 — Test-set scoreboard

Finalists (Ridge + LightGBM) are locked from CV, so now I open the sealed test set.
I refit all 9 on the FULL training history, then predict ONCE on 2022–2023 — the honest,
one-shot exam. I add the persistence baseline as a reference row and mark who beats the bar.

In [4]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

X_test = test[config.MODEL_FEATURES]
y_test = test[config.TARGET_AQI_COL]

BAR_MAE = bar["mae"]   # live persistence bar from Bucket 0 (24.008)

test_rows = []
for name, model in regressors:
    pipe = trainer.make_pipeline(model)
    pipe.fit(X_train, y_train)          # learn from ALL training history
    y_pred = pipe.predict(X_test)       # score ONCE on the sealed test set
    test_rows.append({
        "model":    name,
        "test_mae":  mean_absolute_error(y_test, y_pred),
        "test_rmse": root_mean_squared_error(y_test, y_pred),
        "test_r2":   r2_score(y_test, y_pred),
    })
    print(f"  scored: {name:16s} test_mae={test_rows[-1]['test_mae']:.3f}")

# Add the persistence baseline as the reference line on the scoreboard.
test_rows.append({
    "model":     "Persistence (baseline)",
    "test_mae":  bar["mae"],
    "test_rmse": bar["rmse"],
    "test_r2":   bar["r2"],
})

test_board = pd.DataFrame(test_rows).sort_values("test_mae").reset_index(drop=True)
test_board["beats_bar"] = test_board["test_mae"] < BAR_MAE
test_board["vs_bar"]    = (BAR_MAE - test_board["test_mae"]).round(3)  # +ve = better
test_board.round(3)

  scored: LinearRegression test_mae=23.701
  scored: Ridge            test_mae=23.693
  scored: Lasso            test_mae=25.276
  scored: ElasticNet       test_mae=27.042
  scored: KNN              test_mae=27.694
  scored: DecisionTree     test_mae=35.813
  scored: RandomForest     test_mae=24.099
  scored: XGBoost          test_mae=24.279
  scored: LightGBM         test_mae=23.764


,model,test_mae,test_rmse,test_r2,beats_bar,vs_bar
0,Ridge,23.693,37.929,0.786,True,0.315
1,LinearRegression,23.701,37.932,0.786,True,0.307
2,LightGBM,23.764,37.982,0.786,True,0.244
3,Persistence (baseline),24.008,41.404,0.745,False,0.000
4,RandomForest,24.099,38.327,0.782,False,-0.091
5,XGBoost,24.279,39.033,0.774,False,-0.271
6,Lasso,25.276,39.048,0.774,False,-1.268
7,ElasticNet,27.042,40.577,0.755,False,-3.034
8,KNN,27.694,42.942,0.726,False,-3.686
9,DecisionTree,35.813,57.359,0.511,False,-11.805


## Bucket 4 — Tune the finalists

I tune Ridge and LightGBM on the same time-aware TimeSeriesSplit folds. Ridge has one
real dial (alpha) so I GridSearch it exhaustively; LightGBM has many dials so I
RandomizedSearch its ranges. Then I re-score both tuned winners once on the sealed test set.

In [5]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

cv = trainer.get_cv()

# ---- Ridge: one dial (alpha) -> exhaustive GridSearch ----
# 'model__alpha' = reach INSIDE the pipeline, into the step named 'model', set its alpha.
ridge_grid = {"model__alpha": [0.1, 1.0, 10.0, 50.0, 100.0, 200.0, 500.0]}
ridge_search = GridSearchCV(
    trainer.make_pipeline(Ridge()),
    ridge_grid,
    cv=cv,
    scoring="neg_mean_absolute_error",  # remember: negated, higher = better
    n_jobs=-1,
)
ridge_search.fit(X_train, y_train)

# ---- LightGBM: many dials -> RandomizedSearch over ranges ----
lgbm_dist = {
    "model__n_estimators":   [200, 400, 600, 800],
    "model__num_leaves":     [15, 31, 63, 127],
    "model__learning_rate":  [0.01, 0.03, 0.05, 0.1],
    "model__min_child_samples": [20, 50, 100],
    "model__subsample":      [0.7, 0.8, 1.0],
    "model__colsample_bytree": [0.7, 0.8, 1.0],
}
lgbm_search = RandomizedSearchCV(
    trainer.make_pipeline(LGBMRegressor(random_state=config.RANDOM_STATE, n_jobs=-1, verbose=-1)),
    lgbm_dist,
    n_iter=25,                          # 25 random combos, not the full grid
    cv=cv,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    random_state=config.RANDOM_STATE,   # makes the random search reproducible
)
lgbm_search.fit(X_train, y_train)

# ---- Re-score both tuned winners ONCE on the sealed test set ----
tuned = [("Ridge (tuned)", ridge_search), ("LightGBM (tuned)", lgbm_search)]
for name, search in tuned:
    y_pred = search.best_estimator_.predict(X_test)
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = root_mean_squared_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    print(f"{name:18s} best_cv_mae={-search.best_score_:.3f}  "
          f"test_mae={mae:.3f}  test_rmse={rmse:.3f}  test_r2={r2:.3f}")
    print(f"                   best params: {search.best_params_}\n")

Ridge (tuned)      best_cv_mae=25.818  test_mae=23.693  test_rmse=37.929  test_r2=0.786
                   best params: {'model__alpha': 0.1}

LightGBM (tuned)   best_cv_mae=25.977  test_mae=23.735  test_rmse=38.048  test_r2=0.785
                   best params: {'model__subsample': 1.0, 'model__num_leaves': 63, 'model__n_estimators': 200, 'model__min_child_samples': 50, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.7}



## Bucket 5 — Crystallize + verify

The roster and CV bake-off are proven, so I freeze them into ModelTrainer and
re-run the leaderboard from the imported module. If it matches my Bucket 2 numbers,
the crystallized code is faithful and Session 6 is sealed.

In [6]:
trainer = ModelTrainer()
roster = trainer.regression_roster()

leaderboard = trainer.run_bakeoff(roster, X_train, y_train, config.REGRESSION_SCORING)
print("regression winner (CV):", leaderboard.iloc[0]["model"])
leaderboard.round(3)

regression winner (CV): Ridge


,model,cv_mae,cv_rmse,cv_r2
0,Ridge,25.820,41.016,0.764
1,LinearRegression,25.824,41.018,0.764
2,LightGBM,26.047,41.232,0.762
3,RandomForest,26.602,41.793,0.755
4,XGBoost,27.084,42.766,0.744
5,Lasso,27.759,42.911,0.744
6,ElasticNet,29.747,44.601,0.724
7,KNN,30.770,47.206,0.691
8,DecisionTree,38.612,61.476,0.473
